# 02 Error Analysis for Function Calling

In the previous notebook we have introduced the Evaluation process, now we will perform some error analysis on the outputs generated by our finetuned model.

🚀 Before then, let's load our data with predictions and useful functions.

In [ ]:
import ast
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import warnings

from utils.metrics import calculate_metric
from utils.metric_types import Function, Parameters
from sklearn.metrics import confusion_matrix
warnings.simplefilter(action='ignore')

## Load Data

In [ ]:
# Load prediction data
df = pd.read_csv("assets/prediction_with_metrics.csv")

In [ ]:
# Fix formats
df['ground_truth'] = df['ground_truth'].apply(ast.literal_eval)
df['prediction'] = df['prediction'].apply(ast.literal_eval)

# Drop not useful columns
df = df.drop(columns=['id', 'explicitness', 'emotional_state'])

In [ ]:
# Load the function specifications - this file includes all the details for each function we have to conduct the evaluation on
with open('../Data/function_spec') as f:
    raw_function_specs = json.load(f)

In [ ]:
function_specs = {
    raw_function_spec['function']['name']: raw_function_spec['function']
    for raw_function_spec in raw_function_specs
}

In [ ]:
def simple_metrics(ground_truth, model_answer, functional_only=True):
    metric_dict = calculate_metric(
        function_specs=function_specs,
        expected=Function(name=ground_truth['function_name'], parameters=Parameters(ground_truth['parameters'])),
        functions_from_slm=[Function(name=model_answer['function_name'], parameters=Parameters(model_answer['parameters']))],
        tokens_count=0,

        start_timestamp=None,
        first_token_timestamp=None,
        last_token_timestamp=None
    ).model_dump()

    if functional_only:
        non_functional_metrics = ['secs_to_first_token', 'secs_to_last_token', 'tokens_per_second']
        metric_dict = {key: value for key, value in metric_dict.items() if key not in non_functional_metrics}
        return metric_dict

    else:
        return metric_dict

## Error Analysis
Error analysis is crucial for evaluating Small Language Models (SLMs) because it helps **identify weaknesses**, **improve model performance**, and ensure **reliability** in real-world applications. By analyzing errors we are able to identigy biases or unintended model behaviours, whist highlighting how well the model generalizes across different inputs. It's relevant not only in terms of model performance, but also for optimizing real-world impact in fixing critical errors that could be costly otherwise.

Therefore, error analysis is not just about finding what’s wrong—it’s about systematically improving an SLM to make it more accurate, reliable, and fair. It plays a key role in debugging, optimizing, and deploying models that perform well in real-world scenarios.

Let's start with only one function example:

In [ ]:
def get_sample_function_gt_pred(
        ground_truth: pd.Series,
        predicted: pd.Series,
        function_name: str) -> list:
    """
    Get the ground truth and predicted values for a given function name

    Args:
    -----
    ground_truth: pd.Series
        List of ground truth values
    predicted: pd.Series
        List of predicted values
    function_name: str
        The function name to filter

    Returns:
    --------
    gts: list
        List of ground truth values for the given function name
    preds: list
        List of predicted values for the given function name
    """
    gts = []
    preds = []
    for i in range(len(ground_truth)):
        if ground_truth[i]['function_name'] == function_name:
            gt = 1
        else:
            gt = 0
        if predicted[i]['function_name'] == function_name:
            pred = 1
        else:
            pred = 0
        gts.append(gt)
        preds.append(pred)

    return gts, preds

In [ ]:
def error_analysis_cm(df: pd.DataFrame, function_name: str) -> None:
    """
    Creates a graph comparing ground truth vs predictions for the given function name

    Args:
    -----
    df: pd.DataFrame
        The dataframe containing the ground truth and predicted values
    function_name: str
        The function name to analyze

    Returns:
    --------
    None
    """
    gts, preds = get_sample_function_gt_pred(df['ground_truth'].to_list(), df['prediction'].to_list(), function_name)
    cm = confusion_matrix(gts, preds)
    # Plot confusion matrix
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap="viridis", xticklabels=["Pred 0", "Pred 1"], yticklabels=["True 0", "True 1"], vmax=10)
    plt.xlabel("Predicted")
    plt.ylabel("Ground Truth")
    plt.title(f"Ground Truth vs Predicted for {function_name}")
    plt.show()


We will now select a single functio name and analyze the performance of the ground truth vs the predicted name. Let's take `set_volume` as an example: 
- in 77 cases it has been predicted correctly (`Pred==1 AND True==1`)
- in 3 cases it predicted another function name, when it should've been set_volume instead (`Pred==0 AND True==1`)
- The 320 cases are all the other cases when it was not predicted as `set_volume` and it should not have been `set_volume` (`Pred==0 AND True==0`)

In [ ]:
error_analysis_cm (df, 'set_volume')

If you want to see other function names results, change the parameter `set_volume` with another (included in the function specifications) of your preference!

Aggregate level for all the predictions:

In [ ]:
def error_analysis_func_full(df: pd.DataFrame) -> None:
    """
    Creates a graph comparing ground truth vs predictions for the function names.
    It shows also all the extra generated functions names

    Args:
    -----
    df: pd.DataFrame
        The dataframe containing the ground truth and predicted values

    Returns:
    --------
    None
    """
    # Apply to the entire dataset
    temp_df = df[['ground_truth', 'prediction']].copy()

    # Extract function names from dictionaries
    temp_df["ground_truth"] = temp_df["ground_truth"].apply(lambda x: x["function_name"])
    temp_df["prediction"] = temp_df["prediction"].apply(lambda x: x["function_name"])


    gts = temp_df["ground_truth"].to_list()
    preds = temp_df["prediction"].to_list()

    common = set([item for item in gts])
    extra_pred = set([item for item in preds if item not in common])

    # Create labels and put extra predictions at the end (for better visualization)
    x_func_labels = list(common) + list(extra_pred)
    y_func_labels = list(set(gts))

    # Compute confusion matrix
    cm = confusion_matrix(temp_df["ground_truth"], temp_df["prediction"], labels=x_func_labels)

    # Drop rows where all values are 0
    # Identify rows that are all zeros
    non_zero_rows = np.any(cm != 0, axis=1)

    # Filter out rows that are all zeros
    filtered_cm = cm[non_zero_rows, :]

    # Plot confusion matrix heatmap
    plt.figure(figsize=(6,5))
    sns.heatmap(filtered_cm, annot=True, fmt='d', cmap="viridis", xticklabels=x_func_labels, yticklabels=y_func_labels, vmax=10)
    plt.xlabel("Predicted")
    plt.ylabel("Ground Truth")
    plt.title("Function Names - Ground Truth vs Predicted")
    plt.show()

In [ ]:
error_analysis_func_full(df)

🤔 __Quick Question__: Do you notice anything weird in the above graph? What is it and what do you think caused it?


<details>
    <summary>Answer: Click to show</summary>
    
There are some predicted functions that are NOT in the listed ground truth function names. The model has created some function names out of nowhere: there are hallucinations. In our case, they are: `change_volume` and `adjust_rack_tension` as they are not in the given ground truth. Let's focus our analysis in understanding where this error could originate from and what should we do to solve it.

![graph](images/graph.png)

    
</details>

Check for hallucinations:

In [ ]:
# Extract all function names from ground truth
fn_names = []

for fn_name, _ in list(function_specs.items()):
    fn_names.append(fn_name)

In [ ]:
# Compute hallucinated function names
hallucinated_fn_names = [fc['function_name'] for fc in df['prediction'].to_list() if fc['function_name'] not in fn_names]

n_hallucinated_fn = len(hallucinated_fn_names)

print(
    f"Number of hallucinated function names: {n_hallucinated_fn}, out of {len(df['prediction'])} times, which is {n_hallucinated_fn / len(df['prediction']) * 100:.2f}%"
)

From the above analysis, we notice that the hallucinations occur with the `adjust_volume` function mainly. Let's do a bit of exploratory analysis on the training data used to finetune our language model.

## Explore and Focus on the output of a Targeted Function

In [ ]:
# Choose a function name to analyze
fn = 'adjust_volume'

# Sample data for the function name
sample_df = df[df['name']==fn]
sample_df = sample_df[['prompt', 'ground_truth', 'prediction']].reset_index(drop=True)

In [ ]:
sample_df

In [ ]:
sample_results = sample_df.apply(lambda row: pd.Series(simple_metrics(row['ground_truth'], row['prediction'])), axis=1)

# Add the non-function inference metrics
sample_results[['secs_to_first_token', 'secs_to_last_token', 'tokens_per_second']] = df[df['name']==fn][['time_to_first_token_seconds', 'total_time_seconds', 'tokens_per_second']].reset_index(drop=True)

In [ ]:
sample_results

In [ ]:
# Metrics for the adjust_volume function
print(f'Mean for {fn}:')

pd.concat(
    [
        sample_results.mean().rename('mean')
    ],
    axis=1,
)

If looking only at the functional metrics, we notice that when it is the function being called it is generally well performing, but it struggles in generating the correct arguments for the parameters

In [ ]:
for func in raw_function_specs:
    if func['function']['name'] == fn:
        print(json.dumps(func, indent=4))


From the `adjust_volume` specifications, we notice that the parameters have to be objects and the type it accepts is an integer value. Let's analyze now where it makes a mistake in the parameters:

In [ ]:
sample_df

In [ ]:
def highlight_mismatches(row):
    """
    Compare dictionaries in 'ground_truth' and 'prediction' columns.
    Return a dictionary highlighting mismatched keys.
    """
    gt, pred = row["ground_truth"], row["prediction"]

    mismatches = {k: (gt[k], pred[k]) for k in gt if gt[k] != pred.get(k)}

    return mismatches if mismatches else None  # Return None if no mismatch


In [ ]:
sample_df["mismatches"] = sample_df.apply(highlight_mismatches, axis=1)

print(f'There are in total {len(sample_df[sample_df["mismatches"].notna()])} mismatches for the function {fn}.')

In [ ]:
mismatches_df = sample_df[sample_df["mismatches"].notna()]

mismatches_df.head(10)

Majority have errors in the parameters, but there are also cases in which the function name called is the wrong one, they are:

In [ ]:
print('User query: ', sample_df['prompt'].loc[24])

sample_df[sample_df["mismatches"].notna()]['mismatches'].loc[24]

🤔 __Let's think about it!__: Why do you think this is happening?

<details>
    <summary>Answer: Click to show</summary>
    
The verb "turn up" was not very clear for the model that it needed to be linked to the `adjust_volume` function, and it has created `create_function` which did not exist in the ground truths. However, interestingly, for the parameter values, it seems like the ground truth was also inaccurate: it has a volume increment which cannot be inferred from the user query --> error analysis is useful also to understand if there are some errors in the ground truth!

    
</details>


In [ ]:
print('User query: ', sample_df['prompt'].loc[14])

sample_df[sample_df["mismatches"].notna()]['mismatches'].loc[14]

🤔 __Let's think about it!__: What about here? 


<details>
    <summary>Answer: Click to show</summary>
    
In this case, "racket" was confusing for the model as usually we use it in the sense of "tennis racket" instead of "loud noise". A way to solve it would be to include more examples in the training data that used this noun in the semantic of noise, such that the model can familiarize itself with this connotation. On the other hand, the parameter value is indeed the correct one.

    
</details>


The user query they were predicted for is:

In [ ]:
# Function to classify rows based on dictionary keys
def classify_dict(row):
    keys = set(row.keys())
    if keys == {"parameters"}:
        return "parameters"
    elif keys == {"function_name"}:
        return "function_name"
    elif keys == {"parameters", "function_name"}:
        return "function_name, parameters"
    return "other"

# Apply classification
mismatches_df["category"] = mismatches_df["mismatches"].apply(classify_dict)


# Count occurrences of "parameters", "function_name" or both cases
count_series = mismatches_df["category"].value_counts()

# Plot
plt.figure(figsize=(6, 4))
plt.bar(count_series.index, count_series.values)
plt.xlabel("Category")
plt.ylabel("Count")
plt.title(f"Count of Errors by Category in {fn}")
plt.show()

Majority of the time it is only the parameters that are faulty, but there are 2 instances where both parameters and `function_names` are not the same

## Potential next steps:
Now that we have conducted our Evaluation and Error Analysis, we noticed that our finetuned model is quite well performing, but there are still some areas where it could improve. 

### __What would YOU do next?__ 💡

<details>
    <summary>Here are some ideas: Click to show</summary>
    - Futher and more in depth error analysis </br>
    - Check the ground truths once more to understand if there are any mistakes</br>
    - Repeat the error analysis for all functions and see how they differ


</details>


## Learnt Lessons
- Evaluating on an aggregate level is important but it is also important to deep dive in the errors and understand the reason why certain errors occur
- Sometimes, during evaluation, we can identify also errors that are due to the ground truth

## Resources
- [Finetuning SLMs](https://medium.com/@liana.napalkova/fine-tuning-small-language-models-practical-recommendations-68f32b0535ca)
- [Understanding Hallucinations](https://medium.com/@gcentulani/understanding-hallucination-in-llms-causes-consequences-and-mitigation-strategies-b5e1d0268069#:~:text=In%20the%20context%20of%20LLMs%2C%20hallucination%20refers%20to,accurately%20represent%20or%20reason%20about%20the%20real%20world)
- [Other evaluations](05-other-evaluation.ipynb)

